# **Infant Cry Sound Detection**

## **Abstract**

Please find the repository for this project: https://github.com/JayC-SF/COMP-432-Project

Please also make sure to read the [README.md](https://github.com/JayC-SF/COMP-432-Project/blob/main/README.md) to understand how to replicate the code.

## **Introduction**

This project attempts to solve the infant cry sound detection problem. In this modern day, many  

## **2. Methodology**

### **2.1 Preprocessing**


The audio files were preprocessed in to mel spectograms where they needed to be formatted to the same audio lengths. Most of audio lengths were around 10 seconds, however, there were a few samples that were around 6-8 seconds, with a few outliers with only 1-2 seconds. Since the maximum length of audio was 10 seconds, all samples with less durations were padded with empty sound to fit the durations. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/train_dataset_distribution.png" width=400>
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/validation_dataset_distribution.png" width=400>
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/test_dataset_duration_distribution.png" width=400>
</div>

The class imbalance was verified for the train, val and test splits, where for pretty much all splits there were roughly 50/50 InfantCry/Snoring balancing of the classes.


<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/training_class_imbalance.png" width=400>
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/validation_class_imbalance.png" width=400>
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/testing_class_imbalance.png" width=400>
</div>

### **2.2 Models**


This project performs an analysis of 3 deep learning models: Classical CNN baseline, a Convolutional LSTM, and a Resnet model. 

#### **2.2.1 Classical CNN Architecture**

The ClassicCNN is a traditional sequential architecture designed for hierarchical feature extraction from audio spectrograms. 

It consists of a three-layer convolutional backbone where the depth progressively increases (32, 64, and 128 filters), while integrated Batch Normalization for stability and Max-Pooling layers to reduce spatial dimensions. 

The extracted features are flattened into a high-dimensional vector and processed through a 256-unit fully connected layer, which utilizes 30% Dropout during training to provide regularization before final classification.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn_architecture.png">
</div>

#### **2.2.2 Convolutional LSTM (Best Model)**


The Convolutional LSTM follows the same convolution as the Classical CNN, however instead of applying two fully connected layers it goes through a a two layer LSTM with  `hidden_size=256` to parse the temporal aspect of the input. It can be seen later in this report how this piece does increase the performance of the model. A dropout of 0.3 was also added in this model to add regularization.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/CLSTM_architecture.png">
</div>

### **2.2.3 ResNet**


The ResNet architecture is by the slowest model to train in this report. This model consists of 4 layers of Residual Blocks.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/residualblockpng.png" width=400>
</div>

The residual block contains 2 paths, from which 1 contains 2 convolutional layers followed by batch normalization and relu activation functions and the other one contains only 1 convolutional layer and a single batch norm. The output of those two paths are then merged by adding the results of both paths and a final ReLU activation function is applied.

The architecture of the resnet model is depicted as below, demonstrating 4 layers of residual blocks.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/ResNetArchitecture.drawio.png" width=1000>
</div>

#### **2.2.4 ML Pipeline**

Every model above have been trained with a baseline set of hyper parameters. They were then evaluated with their respective metrics like losses, accuracies (val & train) over epochs. The best epoch for the models. From those evaluations, a range of hyper parameters on learning rates and weight decay were determined to perform the hyper parameter tuning. 

The library `optuna` was leveraged for the hyper parameter tuning. This helped performing some pruning for some searches that were not relevant and helped speed up the fine tuning of the models.

## **3. Experimental Setup**


### **3.1 ICSD Dataset**


In this experiment, the ICSD dataset has been used to train all the models. The ICSD dataset consists of ~14, 000 audio samples with mainly 10 seconds duration. A few samples are not of the same durations where some are around 1-2 seconds with a few more that are 6-9 seconds. 

The total dataset consists of 3 main components: real strong, synthetic strong, and weak labeling.

**Real strong labeling**: Consists of real data that are strongly labeled where the exact time of the infant cry is recorded in the metadata.

**Syntehtic strong labeling**: Consists of synthetic (real infant cries merged with Urban sounds) data that are strongly labeled where the exact time of the infant cry is recorded in the metadata.

**Weak**: Consists of real data where the exact time of the infant cry is not recorded but its classification is noted as either infant cry or snoring.

Since the goal of this project is not to specifically determine the exact time of the infant cry These three components were taken into a single dataset In order to train the models. The ICSD dataset came with a predetermined train validation and test split performed by the owners of the dataset. Since this split was performed by current researchers it is assumed that the dataset does not contain any leaks between each split.

The dataset was preprocessed using the `librosa` library. Every audio samples were converted into mel spectograms. The mel spectogram convertion used a sample rate of 16000, 128 mels, and 512 hop length. 

| Parameter | Value | Explanation |
| :--- | :--- | :--- |
| **Sample Rate (`sr`)** | 16000 |The number of digital snapshots taken per second. It determines the maximum frequency captured (Nyquist frequency). |
| **Mel Bands (`n_mels`)** | 128 | The number of frequency bins on the vertical axis, scaled to match human auditory perception of pitch. |
| **Hop Length (`hop_length`)** | 512 | The number of audio samples the analysis window "steps" forward for each frame; defines the temporal density. Equivalent to taking snapshots of the audio every 23 milliseconds|

After being converted into mel spectograms, the samples were then normalized to a range of [0,1] using a [Min-Max Scaling Normalization](https://en.wikipedia.org/wiki/Feature_scaling#Rescaling_(min-max_normalization)). 


$$x'=\frac{x-min(x)}{max(x)-min(x)}$$

There were three main reasons the dataset was normalized:
1. Potentially better convergence to avoid exploding or vanishing gradients. 
2. Avoids providing links of loud sounds to crying sounds, but instead decode the texture and curves of an infant cry.
3. Combats sensitivity of ReLu activation function since the positive side of the activation function is a linear function.


After the first conversion to mel spectograms, the data is gathered together by either truncating or padding the audio files so they have the same time dimension length (313).

From the records, the data was never really truncated but only padded with empty sound (0-value) for the short duration ones.
The dataset is then converted into a single .npz file to contain all of the mel spectrograms into a single file. The conversion into a single file was to facilitate loading the dataset for training models.


### **3.2 Model Configurations**


An [Orchestrator](https://github.com/JayC-SF/COMP-432-Project/blob/main/src/train/orchestrator.py) class was created for reusability of the train, validation, and test pipelines onto the models.

All models were trained with early stopping and a patience of 15 to provide them a chance to get out of local minimums. They all shared Adam W optimizer as it was determined to be probably the better option as the optimizer for these models. The scheduler was used as well to determine the learning rates of the training. Not all models shared the same notably the Resnet model use the cosine annealing scheduler for the training.  The Cross Entropy Loss function is used even though the binary classification models are trained since the training pipeline can be extended for multi class classification eventually for further projects.

The training pipeline saves the history of the orchestrator in order to continue the training in case the training is halted. The history saved includes the training and validation losses, accuracies, optimizer, scheduler, best model weights, the current model, early stopping count and epoch.

Note as well as mentioned earlier The models were also fine tuned based on learning rate and weight decay only. The Optuna Library was used to perform the fine tuning of these hyperparameters. Each models are trained with 15 trials where some trials were pruned as some training instances were found to have better potential. This was particularly useful since a great search would have been too expensive to train in order to the hyperparameters. Additionally the Optina library took care of selecting the hyperparameters over a continuous interval.

#### **3.2.1 Classic CNN Hyperparameters**

The Classic CNN baseline model hyperparameters can be found in the table below. The learning rate chosen for this baseline was simply a classic baseline LR. We also add a light touch of weight decay to prevent overfitting of the model as well as a 0.3 or dropout rate to help with regularization. Note the scheduler patience of 5 to help converging the learning rate on validation loss plateaus.

| Hyperparameter | Value |
| :--- | :--- |
| **Optimizer** | AdamW |
| **Scheduler** | ReduceLROnPlateau |
| **Scheduler Patience** | 5 |
| **Scheduler Factor** | 0.1 |
| **Learning Rate** | 1e-3 |
| **Batch Size** | 64 |
| **Patience** | 15 |
| **Loss Function** | CrossEntropyLoss |
| **Dropout Rate** | 0.3 |
| **Weight Decay** | 5e-5 |
| **Input Shape** | (12284, 128, 313) |


After studying the training and validation losses, it was determined that the following hyperparameter ranges needed to be employed for the fine tuning.


| Hyperparameter | Low Bound | High Bound |
| :--- | :--- | :--- |
| **Learning Rate** | 1e-5 | 1e-2 |
| **Weight Decay** | 1e-6 | 1e-1 |

#### **3.2.2 Convolutional LSTM Hyperparameters**


Since LSTMs use hidden layers, the potential of vanishing or exploding gradients were possible and therefore a smaller learning rate was potentially a safer choice. This is especially the case since the model has to train through 313 time dimension vector for each sample giving a probable chance of having an exploding gradient if the learning rate was too high. 

Additionally, the learning rate value allows the model to make precise and small adjustements to the gates (input, forget, and output gates), without overshooting the delicate logic required to remember long-term dependencies.

| Parameter | Value |
| :--- | :--- |
| **Optimizer** | AdamW |
| **Scheduler** | ReduceLROnPlateau |
| **Scheduler Patience** | 5 |
| **Scheduler Factor** | 0.1 |
| **Learning Rate** | 5e-4 |
| **Batch Size** | 64 |
| **Patience** | 15 |
| **Loss Function** | CrossEntropyLoss |
| **Dropout Rate** | 0.3 |
| **Weight Decay** | 5e-4 |
| **Input Shape** | (12284, 128, 313) |

On hindsight, the hyperparameter search should've been a little more narrow. The justification for this statement is because from some research the LSTM models tend to be very sensitive to high learning rates and since the hyperparameter ranges for the learning rate and weight decay are still quite large it provided some worse results than the baseline model. Also as it will be seen later in this report it can be seen that at roughly epoch 20 The model starts to overfit and therefore a higher weight decay could have helped against the overfitting.

| Hyperparameter | Low Bound | High Bound |
| :--- | :--- | :--- |
| **Learning Rate** | 1e-5 | 1e-2 |
| **Weight Decay** | 1e-6 | 1e-1 |

#### **3.2.3 ResNet Hyperparameters**

We use a smaller learning rate for the Resnet architecture since it is a deeper network and the search space consists of more peaks and valleys deeper the network is. Since the model also has a lot more convolutional layers, we wanted to avoid the model to overfit too quickly and therefore programmed a higher weight decay compared to the other baselines.

This model uses the cosine scheduler in order to start with a higher learning rate and using the cosine function to reduce the learning rate as the epochs increase.

| Parameter | Value |
| :--- | :--- |
| **Optimizer** | AdamW |
| **Scheduler** | CosineAnnealingLR |
| **T_max** | 40 |
| **Learning Rate** | 1e-4 |
| **Batch Size** | 64 |
| **Patience** | 15 |
| **Loss Function** | CrossEntropyLoss |
| **Dropout Rate** | 0.3 |
| **Weight Decay** | 1e-3 |
| **Input Shape** | (12284, 128, 313) |

Since the baseline model was still very good the learning rate range to explore stayed in the same ranges as The model had the cosine annealing to help with the later stages of the training. As for the weak decay the Resnet architecture tend to overfit very quickly which was the case for the baseline model, and therefore a larger weight decay was explored to avoid the early overfitting.

On hindsight, the `T_max` hyperparameter should have been changed for smaller value since the baseline model converged at roughly 25 epochs.



| Hyperparameter | Low Bound | High Bound |
| :--- | :--- | :--- |
| **Learning Rate** | 1e-5 | 8e-4 |
| **Weight Decay** | 1e-4 | 5e-2 |

## **4. Experimental Results**

### **4.1 Baseline Classic CNN Model**

You can find below the metrics found for the baseline.

| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 21 |
| **Total epochs** | 36 |
| **Best Epoch Training Loss** | 0.04641247166060481 |
| **Best Epoch Validation Loss** | 0.14230672136472466 |
| **Best Epoch Training Accuracy** | 97.842722% |
| **Best Epoch Validation Accuracy** | 95.563771% |
| **Test Loss** |   0.0810 |
| **Test Accuracy** |  97.0425% |

You can see here we have training and validation loss graph over the epochs. As it can be noted at roughly epoch 15 there is a very big spike in validation loss  that further goes down later on on further epochs. This spike also is reflected in the accuracy graph.  We can also note the big difference of the first epoch from the training loss and devalidation loss suggetsing that the model is capable of learning potentially very fast on the data. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/train_val_losses_graph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/baseline_accs_graph.png">
</div>

```plain
              precision    recall  f1-score   support

     Snoring       0.95      0.99      0.97       539
   InfantCry       0.99      0.95      0.97       543

    accuracy                           0.97      1082
   macro avg       0.97      0.97      0.97      1082
weighted avg       0.97      0.97      0.97      1082
```

We can also find the confusion matrix for the model. And it is as expected that the model is capable of recognizing more infant cries then simply snoring sounds. This is potentially because some sounds that were mixed with the snoring audios did contain some loud noises that were probably confusing the model. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/conf_matrix.png">
</div>

The test loss and training accuracy are 0.0810 and 97.0425% respectively


### **4.2 Fine Tuned Classic CNN Model**

For the fine tuning we can see parameters selected As we can see some trials were pruned while others were completed. 

The best trial validation score was of 0.1331 With a learning rate of 0.0003899009112596919 and a weight decay of 7.783037147953763e-05.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/classical_cnn_optuna_table.png">
</div>

| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 16 |
| **Total epochs** | 31 |
| **Best Epoch Training Loss** | 0.04641247166060481 |
| **Best Epoch Validation Loss** | 0.13305390068431439 |
| **Best Epoch Training Accuracy** | 98.477695% |
| **Best Epoch Validation Accuracy** | 95.563771% |
| **Test Loss** |   0.0727 |
| **Test Accuracy** |  98.1516% |

The fine tuned model converged 5 epochs faster with a total number of epochs of 31 and found a better validation loss as well.

We can see even with the better model found from fine tuning the interesting peak on the validation loss at epoch 15. It seems the model potentially tries to get away from a local minimum or get stuck on saddle point for a while.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/fined_tuned_losses_graph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/classical_cnn/fined_tuned_accs_graph.png">
</div>

We can also note an increase on the classification report where some values have increased to 0.99 and 0.98. Overall on average the values increase quite well and not only one specific value increased a good majority of all the values did increase together.

```plain
                 precision  recall    f1-score  support

     Snoring       0.97      0.99      0.98       539
   InfantCry       0.99      0.97      0.98       543

    accuracy                           0.98      1082
   macro avg       0.98      0.98      0.98      1082
weighted avg       0.98      0.98      0.98      1082
```

We can also note an increase on the test accuracy a jump from ~97% to ~98.1%. Not a major jump but for sure a satisfying one in less epochs. 

### **4.3 Baseline Convolutional LSTM Model**


You can find the baseline CLSTM results below. 


| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 19 |
| **Total epochs** | 34 |
| **Best Epoch Training Loss** | 0.02332739161890945 |
| **Best Epoch Validation Loss** | 0.07456372386316275 |
| **Best Epoch Training Accuracy** | 99.226636% |
| **Best Epoch Validation Accuracy** | 97.319778% |
| **Test Loss** |   0.0296 |
| **Test Accuracy** |  98.9834% |

Compared to the previous model this model is able to achieve a higher accuracy and losses overall in training, validation and test. This potentially demonstrates as well how the LSTM component of the model is capable of parsing through the time of the input and inferring better than an FC layer. It is notably very interesting since both the CN baseline and the convolutional LSTM model had the same convolutional component in their architecture. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/train_val_losses_graph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/train_val_accs_graph.png">
</div>

Something to also be expected is that the snoring classification can get confused specifically depending on the urban sounds that were mixed on the synthetic audio. This is something to note as it is also something both models with the CNN and the LSTM struggle with. However the convolutional LSTM model is still capable of achieving better results as well.


<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/conf_matrixoutput.png">
</div>

### **4.4 Fine Tuned Convolutional LSTM Model**


As mentioned earlier the ranges for the hyperparameter tuning were not as optimal as it should have been . On hindsight, it would have been better to actually have a shorter range on the learning rate more specifically close to the baseline model and potentially a little bit of a higher weight decay to combat for the overfitting. The ranges were too large and therefore the Optuna library was more or less guessing a path towards a better result. This it led to overall worse results on the best model found from the fine tuning.


| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 9 |
| **Total epochs** | 24 |
| **Best Epoch Training Loss** | 0.08058485605164717 |
| **Best Epoch Validation Loss** | 0.08138599825158797 |
| **Best Epoch Training Accuracy** | 96.987952% |
| **Best Epoch Validation Accuracy** | 97.042514% |
| **Test Loss** |   0.0487 |
| **Test Accuracy** | 98.6137% |

Best Trial Validation Score: 0.0814
<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/fine_tuning_table.png">
</div>

The best hyperparameters found or 0.00026016923231189166 for the learning rate and 1.3170754520941358e-06 for the weight decay.


We can see as well for the fine tuned model it goes through a lot of spikes for the validation loss and accuracy.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/finetune_losses_garph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/finetune_accs_graph.png">
</div>

We can see as well a bit of a dip on the classification report. We can note that some metrics are no longer perfect while some overall others are just slightly lower cmpared to the baseline model. The classification report still provides a better result than the Classical CNN baseline and fine tuned model. 

```plain
              precision    recall  f1-score   support

     Snoring       0.98      0.99      0.99       539
   InfantCry       0.99      0.98      0.99       543

    accuracy                           0.99      1082
   macro avg       0.99      0.99      0.99      1082
weighted avg       0.99      0.99      0.99      1082
```

As for the confusion matrix we can note some small differences as well. The difference is mainly on the classification of the infant cry where the model gets slightly more confused with a soring sample and classifies it as an infant cry actually. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/finetune_conf.png">
</div>

### **4.5 Baseline Resnet Model**

You can find the baseline Resnet results below.
We can already see the model converges very very fast. This is why This experiment attempted to provide a higher weight decay in order to combat overfitting. It is possible that the nature of the problem is too simple for this model and therefore could cause a higher potential for overfitting.

We can see the matrix are still very good, however, it's still falls short to the convolutional LSTM model shown above.



| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 8 |
| **Total epochs** | 23 |
| **Best Epoch Training Loss** | 0.08472675779420132 |
| **Best Epoch Validation Loss** | 0.12412985875624165 |
| **Best Epoch Training Accuracy** | 97.281016% |
| **Best Epoch Validation Accuracy** |  95.563771% |
| **Test Loss** |   0.0988 |
| **Test Accuracy** |   96.6728% |

We can see the validation loss and accuracy or very jittery over the epochs. It is possible the learning rate may be a little too high . Note that the model was also trained with a Tmax value of 40 and therefore the scheduler does not necessarily have enough epochs to bring the learning rate low enough so the validation and accuracy values are less jittery and converge slowly.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/resnet/train_val_losses_graph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/resnet/train_val_accs_graph.png">
</div>

We can find classification report where all of the metrics are relatively consistent across all fields, It is possible that the resnet model could have performed a lot better given the proper hyperparameters for the training. 

```plain
              precision    recall  f1-score   support

     Snoring       0.97      0.96      0.97       539
   InfantCry       0.96      0.97      0.97       543

    accuracy                           0.97      1082
   macro avg       0.97      0.97      0.97      1082
weighted avg       0.97      0.97      0.97      1082
```

We can see the confusion matrix is also relatively stable the difference between the confusions on both classes are somewhat pretty close. Nonetheless the results are still quite satisfactory. 

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/resnet/conf.png">
</div>

### **4.6 Fine Tuned Resnet Model**

Please find the fine tune results below. 

Best Trial Validation Score: 0.0804

Best learning rate: 0.0007938990750990815

Best weight decay: 0.001832465296770582
<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/resnet/fine_tuning_table.png">
</div>

We can see we managed to increase the total number of epochs for the training. And this way the model was overfitting slower compared to the baseline model. It also was capable of achieving a Higher level of Metrics especially in the validation and test accuracy/losses.

| Metrics | Value |
| :--- | :--- |
| **Best Epoch** | 19 |
| **Total epochs** | 34 |
| **Best Epoch Training Loss** |0.011266060391558048 |
| **Best Epoch Validation Loss** | 0.080359036099448 |
| **Best Epoch Training Accuracy** | 99.715077% |
| **Best Epoch Validation Accuracy** |  97.134935% |
| **Test Loss** |    0.0573 |
| **Test Accuracy** |  97.5046% |

We can still see that the validation plot show a lot of ups and downs. As mentioned earlier the model should have been trained with potentially a lower T Max value. A first hand training should have been done to evaluate the effect of the T Max. Additionally, The T Max value should probably have been included during the hyperparameter search with the learning rate and weight decay.

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/finetune_losses_garph.png">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/clstm/finetune_accs_graph.png">
</div>

```plain
              precision    recall  f1-score   support

     Snoring       0.96      0.99      0.98       539
   InfantCry       0.99      0.96      0.97       543

    accuracy                           0.98      1082
   macro avg       0.98      0.98      0.98      1082
weighted avg       0.98      0.98      0.98      1082
```

<div align="center">
    <img src="https://raw.githubusercontent.com/JayC-SF/COMP-432-Project/main/images/resnet/finetune_conf.png">
</div>

## **Conclusions**

The CLSTM model was the best model from this experiment. There are a lot of decisions that would have been performed differently on hindsight. Notably,

1. An narrower learning rate search for the LSTM model
2. Adding the Tmax value during the fine tuning of the Resnet architecture. 
3. Use another data set to Provide another basis of truth for the potential of these models. 

Nonetheless, the results ended up being quite exceptional. And this will definitely serve a great purpose for the scientific community. I will myself be trying this experiment on my own soon to come son in due in May 16th 😊

## **References**
He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 770-778.

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. Neural Computation, 9(8), 1735-1780.

Kingma, D. P., & Ba, J. (2014). Adam: A method for stochastic optimization. arXiv preprint arXiv:1412.6980.

Hershey, S., et al. (2017). CNN architectures for large-scale audio classification. IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP), 131-135.

Sainath, T. N., Vinyals, O., Senior, A., & Sak, H. (2015). Convolutional, Long Short-Term Memory, fully connected Deep Neural Networks. IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP), 4580-4584.

Sainath, T. N., Vinyals, O., Senior, A., & Sak, H. (2015). Convolutional, Long Short-Term Memory, fully connected Deep Neural Networks. IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP), 4580-4584.

Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining, 2623-2631.

Loshchilov, I., & Hutter, F. (2016). SGDR: Stochastic gradient descent with warm restarts. arXiv preprint arXiv:1608.03983.
